Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/MachineLearningNotebooks/how-to-use-azureml/work-with-data/datadrift-tutorial/datadrift-quickdemo.png)

# Analyze data drift in Azure Machine Learning datasets 

In this tutorial, you will setup a data drift monitor on a weather dataset to:

&#x2611; Analyze historical data for drift

&#x2611; Setup a monitor to recieve email alerts if data drift is detected going forward

If your workspace is Enterprise SKU, view and exlpore the results in the Azure Machine Learning studio. The video below shows the results from this tutorial (spoiler alert). 

## Prerequisites
If you are using an Azure Machine Learning Compute instance, you are all set. Otherwise, go through the [configuration notebook](../../../configuration.ipynb) first if you haven't already established your connection to the AzureML Workspace.

In [14]:
# Check core SDK version number
import azureml.core

print('SDK version:', azureml.core.VERSION)

SDK version: 1.0.65


## Initialize Workspace

Initialize a workspace object from persisted configuration.

In [15]:
from azureml.core import Workspace

ws = Workspace.from_config()
ws

Workspace.create(name='Azure-ML', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG')

## Setup target and baseline datasets

Setup the baseline and target datasets. The baseline will be used to compare each time slice of the target dataset, which is sampled by a given frequency. For further details, see [our documentation](http://aka.ms/datadrift). 

The next few cells will:
  * get the default datastore
  * upload the `weather-data` to the datastore
  * create the Tabular dataset from the data
  * add the timeseries trait by specifying the timestamp column `datetime`
  * register the dataset
  * create the baseline as a time slice of the target dataset
  * optionally, register the baseline dataset
  
The folder `weather-data` contains weather data from the [NOAA Integrated Surface Data](https://azure.microsoft.com/services/open-datasets/catalog/noaa-integrated-surface-data/) filtered down to to station names containing the string 'FLORIDA' to reduce the size of data. See `get_data.py` to see how this data is curated and modify as desired. This script may take a long time to run, hence the data is provided in the `weather-data` folder for this demo.

In [16]:
# use default datastore
dstore = ws.get_default_datastore()

In [17]:
# upload weather data
dstore.upload('training-dataset', 'drift-demo-data', overwrite=True, show_progress=False);

In [19]:
# import Dataset class
from azureml.core import Dataset

# create dataset 
dset = Dataset.Tabular.from_delimited_files(dstore.path('drift-demo-data/training.csv'))
dset = dset.register(ws, 'drift-demo-dataset')

In [20]:
dset = Dataset.get_by_name(ws, 'drift-demo-dataset')

In [21]:
from azureml.core.model import Model
#from azureml.core.resource_configuration import ResourceConfiguration

model = Model.register(model_path='elevation-regression-model.pkl',
                       model_name='elevation-regression-model.pkl',
                       tags={'area': "weather", 'type': "linear regression"},
                       description='Linear regression model to predict elevation based on the weather',
                       workspace=ws,
                       datasets=[(Dataset.Scenario.TRAINING, dset)])

Registering model elevation-regression-model.pkl


In [22]:
from azureml.core import Environment

env = Environment.from_conda_specification(name='deploytocloudenv', file_path='myenv.yml')

In [23]:
from azureml.core.model import InferenceConfig

inference_config = InferenceConfig(entry_script="score.py", environment=env)

In [24]:
from azureml.core.compute import AksCompute, ComputeTarget

# Use the default configuration (you can also provide parameters to customize this).
# For example, to create a dev/test cluster, use:
# prov_config = AksCompute.provisioning_configuration(cluster_purpose = AksCompute.ClusterPurpose.DEV_TEST)
prov_config = AksCompute.provisioning_configuration()

aks_name = 'myaks'
# Create the cluster
try:
    aks_target = ws.compute_targets[aks_name]
except:
    aks_target = ComputeTarget.create(workspace = ws,
                                      name = aks_name,
                                      provisioning_configuration = prov_config)

    # Wait for the create process to complete
    aks_target.wait_for_completion(show_output = True)

Creating...............................................................................................................................................................
SucceededProvisioning operation finished, operation "Succeeded"


In [30]:
from azureml.core.webservice import AksWebservice, Webservice
from azureml.exceptions import WebserviceException

deployment_config = AksWebservice.deploy_configuration(cpu_cores=1, memory_gb=1)
service_name = 'myservice'

service = Model.deploy(ws, service_name, [model], inference_config, deployment_config, aks_target)

service.wait_for_deployment(True)
print(service.state)

Running.....
SucceededAKS service creation operation finished, operation "Succeeded"
Healthy


In [31]:
service.update(collect_model_data=True)

In [41]:
from datetime import datetime, timedelta
from azureml.opendatasets import NoaaIsdWeather

start = datetime.today() - timedelta(days=2)
end = datetime.today()

isd = NoaaIsdWeather(start, end)

df = isd.to_pandas_dataframe().fillna(0)
df = df[df['stationName'].str.contains('FLORIDA', regex=True, na=False)]

X_features = ['latitude', 'longitude', 'temperature', 'windAngle', 'windSpeed']
y_features = ['elevation']

X = df[X_features]
y = df[y_features]

ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2019/month=10/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2019/month=10/part-00122-tid-5528606016710268819-0e3b2550-b4a0-4062-8f9c-d1294ec0c319-1751516.c000.snappy.parquet under container isdweatherdatacontainer
Done.
ActivityCompleted: Activity=to_pandas_dataframe_in_worker, HowEnded=Success, Duration=30209.61 [ms]
ActivityCompleted: Activity=to_pandas_dataframe, HowEnded=Success, Duration=30233.86 [ms]


In [42]:
import json

today_data = json.dumps({'data': X.values.tolist()})

data_encoded = bytes(today_data, encoding='utf8')
prediction = service.run(input_data=data_encoded)
print(prediction)

[[134.89638411725815], [136.2765413926], [134.0429269028778], [130.02811536168986], [117.32105091644638], [131.6687668082826], [136.93005569943776], [18.955920535679127], [19.355806350764325], [29.363107509032446], [29.82354458782595], [30.67700180220632], [30.876944709748912], [144.34684970448754], [146.1874440586229], [147.11402993466956], [-10.987609408152338], [-10.467775026689225], [-5.793737460203857], [-9.287560658889959], [-7.573780550630886], [-6.320437521165331], [-6.8471375821268055], [-3.0928202121897925], [-0.12569707446516531], [-9.472618246397161], [-5.391543723041224], [-5.391543723041224], [13.696472717448302], [-6.251866616919955], [12.182635516731835], [14.22317277840979], [14.22317277840979], [-4.211329355241986], [14.22317277840979], [41.54081329555177], [14.683609857203322], [0.6189443949301108], [-3.6620730359684046], [-3.988830189387272], [-6.622330494194671], [-4.515530250348746], [-4.515530250348746], [-3.5283931105937825], [-1.2879129413732358], [0.5526814127

In [ ]:
#ws.webservices['myservice'].delete()

In [47]:
from azureml.datadrift import DataDriftDetector, AlertConfiguration

services = [service_name]
start = datetime.now() - timedelta(days=2)
feature_list = X_features
alert_config = AlertConfiguration(['cody.peterson@microsoft.com'])

try:
    monitor = DataDriftDetector.create(ws, model.name, model.version, services, frequency='Day', schedule_start=datetime.utcnow() + timedelta(days=1), alert_config=alert_config, compute_target_name='cpu-cluster')
except KeyError:
    monitor = DataDriftDetector.get(ws, model.name, model.version)
    
monitor

2019-10-15 15:57:21,969 - azureml.datadrift._logging._telemetry_logger.datadriftdetector.create - ERROR - DataDriftDetector already exists for model_name: elevation-regression-model.pkl, model_version: 1. Please use DataDriftDetector.get() to retrieve the object - activity_id:7a9761cb-b79b-426c-8e63-13308d4c883b activity_name:datadriftdetector.create activity_type:InternalCall workspace_name:Azure-ML model_name:elevation-regression-model.pkl model_version:1 subscription_id:60582a10-b9fd-49f1-a546-c4194134bba8 workspace_location:southcentralus sdk_version:1.0.65 telemetry_component_name:azureml.datadrift
2019-10-15 15:57:21,970 - azureml.datadrift._logging._telemetry_logger.datadriftdetector.create - ERROR - ActivityCompleted: Activity=datadriftdetector.create, HowEnded=Failure, Duration=416.17 [ms], Exception=KeyError
Traceback (most recent call last):
  File "/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages/azureml/telemetry/activity.py", line 98, in log_activity
    yield a

{'_logger': <_TelemetryLoggerContextAdapter azureml.datadrift._logging._telemetry_logger.azureml.datadrift.datadriftdetector (DEBUG)>, '_workspace': Workspace.create(name='Azure-ML', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG'), '_frequency': 'Day', '_schedule_start': datetime.datetime(2019, 10, 16, 22, 56, 11, 250933, tzinfo=<FixedOffset '+00:00'>), '_schedule_id': None, '_interval': 1, '_state': 'Disabled', '_alert_config': {'email_addresses': ['user@contoso.com']}, '_type': 'ModelBased', '_id': 'fc3cd4b9-a6b1-4981-a095-252a79a10391', '_model_name': 'elevation-regression-model.pkl', '_model_version': 1, '_services': ['myservice'], '_compute_target_name': 'cpu-cluster', '_drift_threshold': 0.2, '_baseline_dataset_id': '00000000-0000-0000-0000-000000000000', '_target_dataset_id': '00000000-0000-0000-0000-000000000000', '_feature_list': None, '_latency': 0, '_name': None, '_client': <azureml.datadrift._restclient.datadrift_client.DataDriftClient o

In [49]:
alert_config = AlertConfiguration(['cody.peterson@microsoft.com'])
monitor.update(alert_config = alert_config)

{'_logger': <_TelemetryLoggerContextAdapter azureml.datadrift._logging._telemetry_logger.azureml.datadrift.datadriftdetector (DEBUG)>, '_workspace': Workspace.create(name='Azure-ML', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG'), '_frequency': 'Day', '_schedule_start': datetime.datetime(2019, 10, 16, 22, 56, 11, 250933, tzinfo=<FixedOffset '+00:00'>), '_schedule_id': None, '_interval': 1, '_state': 'Disabled', '_alert_config': {'email_addresses': ['cody.peterson@microsoft.com']}, '_type': 'ModelBased', '_id': 'fc3cd4b9-a6b1-4981-a095-252a79a10391', '_model_name': 'elevation-regression-model.pkl', '_model_version': 1, '_services': ['myservice'], '_compute_target_name': 'cpu-cluster', '_drift_threshold': 0.2, '_baseline_dataset_id': '00000000-0000-0000-0000-000000000000', '_target_dataset_id': '00000000-0000-0000-0000-000000000000', '_feature_list': None, '_latency': 0, '_name': None, '_client': <azureml.datadrift._restclient.datadrift_client.DataDr

In [50]:
now = datetime.utcnow()
target_date = datetime(now.year, now.month, now.day)
run = monitor.run(target_date, services, feature_list=feature_list, compute_target_name='cpu-cluster')

In [ ]:
run.wait_for_completion(wait_post_processing=True)

In [53]:
results, metrics = monitor.get_output(run_id=run.id)

2019-10-15 16:00:12,954 - azureml.datadrift._logging._telemetry_logger.azureml.datadrift.datadriftdetector - ERROR - Metrics do not exist for model: elevation-regression-model.pkl with version: 1 - workspace_name:Azure-ML model_name:elevation-regression-model.pkl model_version:1 subscription_id:60582a10-b9fd-49f1-a546-c4194134bba8 workspace_location:southcentralus sdk_version:1.0.65 telemetry_component_name:azureml.datadrift


NameError: Metrics do not exist for model: elevation-regression-model.pkl with version: 1

In [56]:
monitor.enable_schedule()

2019-10-15 16:08:35,730 - azureml.datadrift._logging._telemetry_logger.azureml.datadrift.datadriftdetector - ERROR - ActivityCompleted: Activity=enable_schedule, HowEnded=Failure, Duration=14566.66 [ms], Exception=TypeError
Traceback (most recent call last):
  File "/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages/azureml/_logging/chained_identity.py", line 38, in __init__
    super(ChainedIdentity, self).__init__(**kwargs)
TypeError: __init__() got an unexpected keyword argument '_create_in_cloud'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages/azureml/telemetry/activity.py", line 98, in log_activity
    yield activityLogger
  File "/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages/azureml/datadrift/datadriftdetector.py", line 859, in enable_schedule
    schedule = Schedule.get(self.workspace, self._schedule_id)
  File "/home/cody/miniconda3/en

TypeError: __init__() got an unexpected keyword argument '_create_in_cloud'. Found key word arguments: dict_keys(['experiment', '_create_in_cloud']).

## Next steps

  * See [our documentation](https://aka.ms/datadrift) or [Python SDK reference](https://docs.microsoft.com/python/api/overview/azure/ml/intro)
  * [Send requests or feedback](mailto:driftfeedback@microsoft.com) on data drift directly to the team
  * Please open issues with data drift here on GitHub or on StackOverflow if others are likely to run into the same issue